# Stochastic Simulation Algorithm (SSA)

## Required packages

The notebook requires:

- `numpy`
- `matplotlib`
- `gillespy2`

Install GillesPy2 when necessary with:

```bash
pip install gillespy2
```

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import time

import gillespy2
import matplotlib.pyplot as plt
import numpy as np


In [2]:
@dataclass(frozen=True)
class SSAExample:
    """
    Parameters
    ----------
    example_id:
        Integer identifier used to select and label the example.
    D0
    D1
    R_max:
        Largest burst size included in the truncated burst model.

    lam:
        D_r = lam**(r - 1) * D1.
    decay_rate:
        Degradation rate of the proteins.

    t_end:
        Final simulation time. Only the initial and final time points are
        stored because this workflow analyzes the terminal protein count.

    trajectories:
        Default number of independent stochastic trajectories.

    output_filename:
        File name used when saving the terminal protein counts with np.save.

    description:
        Optional human-readable explanation of the parameter set.
    """

    example_id: int
    D0: np.ndarray
    D1: np.ndarray
    R_max: int
    lam: float
    decay_rate: float
    t_end: float
    trajectories: int
    output_filename: str
    description: str = ""

    @property
    def n_states(self) -> int:
        return int(self.D0.shape[0])


def make_generator_matrix(
    n_states: int,
    transitions: Iterable[tuple[int, int, float]],
) -> np.ndarray:
    matrix = np.zeros((n_states, n_states), dtype=float)

    for source, target, rate in transitions:
        if source == target:
            raise ValueError(
                "The transition list must not contain diagonal entries."
            )

        matrix[source, target] = float(rate)

    np.fill_diagonal(matrix, -matrix.sum(axis=1))

    return matrix


## Parameters in the nine examples


| Example | Number of states | `R_max` | `lam` | Protein decay rate | End time | Default trajectories | Output file |
|---:|---:|---:|---:|---:|---:|---:|---|
| 1 | 3 | 20 | 0.10 | 1.0 | 50 | 1,000,000 | `Protein_Number_Burst1.npy` |
| 2 | 3 | 20 | 0.20 | 1.0 | 50 | 1,000,000 | `Protein_Number_Burst2.npy` |
| 3 | 2 | 20 | 0.30 | 1.0 | 50 | 1,000,000 | `Protein_Number_Burst3.npy` |
| 4 | 2 | 20 | 0.30 | 1.0 | 50 | 1,000,000 | `Protein_Number_Burst4.npy` |
| 5 | 5 | 20 | 0.20 | 1.0 | 1,000 | 10,000 | `Protein_Number_Burst5.npy` |
| 6 | 5 | 10 | 0.10 | 1.0 | 1,000 | 10,000 | `Protein_Number_Burst6.npy` |
| 7 | 5 | 10 | 0.05 | 1.0 | 1,000 | 10,000 | `Protein_Number_Burst7.npy` |
| 8 | 10 | 10 | 0.10 | 1.0 | 3,000 | 10,000 | `Protein_Number_Burst8.npy` |
| 9 | 10 | 10 | 0.10 | 1.0 | 10,000 | 10,000 | `Protein_Number_Burst9.npy` |


In [3]:
def build_example_configs() -> dict[int, SSAExample]:
    return {
        1: SSAExample(
            example_id=1,
            D0=np.array(
                [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [1.0, 1.0, 1.0],
                    [1.0, 5.0, 1.0],
                    [1.0, 1.0, 5.0],
                ],
                dtype=float,
            ),
            R_max=20,
            lam=0.1,
            decay_rate=1.0,
            t_end=50.0,
            trajectories=1_000_000,
            output_filename="Protein_Number_Burst1.npy",
            description="Three-state burst model with lambda equal to 0.1.",
        ),
        2: SSAExample(
            example_id=2,
            D0=np.array(
                [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [1.0, 1.0, 1.0],
                    [1.0, 5.0, 1.0],
                    [1.0, 1.0, 5.0],
                ],
                dtype=float,
            ),
            R_max=20,
            lam=0.2,
            decay_rate=1.0,
            t_end=50.0,
            trajectories=1_000_000,
            output_filename="Protein_Number_Burst2.npy",
            description="Example 1 matrices with lambda equal to 0.2.",
        ),
        3: SSAExample(
            example_id=3,
            D0=np.array(
                [
                    [-0.05, 0.05],
                    [0.20, -3.50],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.0, 0.0],
                    [0.0, 2.0],
                ],
                dtype=float,
            ),
            R_max=20,
            lam=0.3,
            decay_rate=1.0,
            t_end=50.0,
            trajectories=1_000_000,
            output_filename="Protein_Number_Burst3.npy",
            description="Two-state model with relatively slow switching.",
        ),
        4: SSAExample(
            example_id=4,
            D0=np.array(
                [
                    [-4.0, 4.0],
                    [6.0, -9.0],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.0, 0.0],
                    [0.0, 2.0],
                ],
                dtype=float,
            ),
            R_max=20,
            lam=0.3,
            decay_rate=1.0,
            t_end=50.0,
            trajectories=1_000_000,
            output_filename="Protein_Number_Burst4.npy",
            description="Two-state model with faster switching.",
        ),
        5: SSAExample(
            example_id=5,
            D0=np.array(
                [
                    [-0.03, 0.03, 0.00, 0.00, 0.00],
                    [0.02, -0.06, 0.04, 0.00, 0.00],
                    [0.00, 0.02, -0.05, 0.03, 0.00],
                    [0.00, 0.00, 0.02, -0.06, 0.04],
                    [0.00, 0.00, 0.00, 0.02, -0.02],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.8, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 2.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 3.0],
                ],
                dtype=float,
            ),
            R_max=20,
            lam=0.2,
            decay_rate=1.0,
            t_end=1_000.0,
            trajectories=10_000,
            output_filename="Protein_Number_Burst5.npy",
            description="Five-state chain with three active burst states.",
        ),
        6: SSAExample(
            example_id=6,
            D0=np.array(
                [
                    [-0.015, 0.015, 0.000, 0.000, 0.000],
                    [0.010, -0.030, 0.020, 0.000, 0.000],
                    [0.000, 0.010, -0.025, 0.015, 0.000],
                    [0.000, 0.000, 0.010, -0.030, 0.020],
                    [0.000, 0.000, 0.000, 0.010, -0.010],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.5, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 1.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 5.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 10.0],
                ],
                dtype=float,
            ),
            R_max=10,
            lam=0.1,
            decay_rate=1.0,
            t_end=1_000.0,
            trajectories=10_000,
            output_filename="Protein_Number_Burst6.npy",
            description="Five-state chain with widely separated burst rates.",
        ),
        7: SSAExample(
            example_id=7,
            D0=np.array(
                [
                    [-0.015, 0.015, 0.000, 0.000, 0.000],
                    [0.010, -0.030, 0.020, 0.000, 0.000],
                    [0.000, 0.010, -0.025, 0.015, 0.000],
                    [0.000, 0.000, 0.010, -0.030, 0.020],
                    [0.000, 0.000, 0.000, 0.010, -0.010],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.01, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.1, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 1.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 10.0],
                ],
                dtype=float,
            ),
            R_max=10,
            lam=0.05,
            decay_rate=1.0,
            t_end=1_000.0,
            trajectories=10_000,
            output_filename="Protein_Number_Burst7.npy",
            description="Example 6 switching matrix with lower burst rates.",
        ),
        8: SSAExample(
            example_id=8,
            D0=np.array(
                [
                    [-0.08, 0.08, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
                    [0.08, -0.09, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
                    [0.00, 0.01, -0.09, 0.08, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
                    [0.00, 0.00, 0.08, -0.16, 0.08, 0.00, 0.00, 0.00, 0.00, 0.00],
                    [0.00, 0.00, 0.00, 0.08, -0.09, 0.01, 0.00, 0.00, 0.00, 0.00],
                    [0.00, 0.00, 0.00, 0.00, 0.01, -0.09, 0.08, 0.00, 0.00, 0.00],
                    [0.00, 0.00, 0.00, 0.00, 0.00, 0.08, -0.16, 0.08, 0.00, 0.00],
                    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.08, -0.09, 0.01, 0.00],
                    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.01, -0.09, 0.08],
                    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.08, -0.08],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 10.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 10.0],
                ],
                dtype=float,
            ),
            R_max=10,
            lam=0.1,
            decay_rate=1.0,
            t_end=3_000.0,
            trajectories=10_000,
            output_filename="Protein_Number_Burst8.npy",
            description="Ten-state model with clustered switching dynamics.",
        ),
        9: SSAExample(
            example_id=9,
            D0=np.array(
                [
                    [-0.050, 0.050, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
                    [0.050, -0.055, 0.005, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
                    [0.000, 0.005, -0.055, 0.050, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
                    [0.000, 0.000, 0.050, -0.055, 0.005, 0.000, 0.000, 0.000, 0.000, 0.000],
                    [0.000, 0.000, 0.000, 0.005, -0.055, 0.050, 0.000, 0.000, 0.000, 0.000],
                    [0.000, 0.000, 0.000, 0.000, 0.050, -0.055, 0.005, 0.000, 0.000, 0.000],
                    [0.000, 0.000, 0.000, 0.000, 0.000, 0.005, -0.055, 0.050, 0.000, 0.000],
                    [0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.050, -0.055, 0.005, 0.000],
                    [0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.005, -0.055, 0.050],
                    [0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.050, -0.050],
                ],
                dtype=float,
            ),
            D1=np.array(
                [
                    [0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 10.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 10.0],
                ],
                dtype=float,
            ),
            R_max=10,
            lam=0.1,
            decay_rate=1.0,
            t_end=10_000.0,
            trajectories=10_000,
            output_filename="Protein_Number_Burst9.npy",
            description="Ten-state model with five paired expression levels.",
        ),
    }


EXAMPLES = build_example_configs()


In [4]:
def print_example_summary(
    examples: dict[int, SSAExample] = EXAMPLES,
) -> None:
    header = (
        f"{'ID':>2}  {'N':>2}  {'R_max':>5}  {'lambda':>7}  "
        f"{'t_end':>8}  {'trajectories':>12}  output"
    )

    print(header)
    print("-" * len(header))

    for example_id, config in examples.items():
        print(
            f"{example_id:>2}  "
            f"{config.n_states:>2}  "
            f"{config.R_max:>5}  "
            f"{config.lam:>7.3g}  "
            f"{config.t_end:>8.1f}  "
            f"{config.trajectories:>12,d}  "
            f"{config.output_filename}"
        )


print_example_summary()

ID   N  R_max   lambda     t_end  trajectories  output
------------------------------------------------------
 1   3     20      0.1      50.0     1,000,000  Protein_Number_Burst1.npy
 2   3     20      0.2      50.0     1,000,000  Protein_Number_Burst2.npy
 3   2     20      0.3      50.0     1,000,000  Protein_Number_Burst3.npy
 4   2     20      0.3      50.0     1,000,000  Protein_Number_Burst4.npy
 5   5     20      0.2    1000.0        10,000  Protein_Number_Burst5.npy
 6   5     10      0.1    1000.0        10,000  Protein_Number_Burst6.npy
 7   5     10     0.05    1000.0        10,000  Protein_Number_Burst7.npy
 8  10     10      0.1    3000.0        10,000  Protein_Number_Burst8.npy
 9  10     10      0.1   10000.0        10,000  Protein_Number_Burst9.npy


In [5]:
def validate_config(config: SSAExample) -> None:
    if config.D0.ndim != 2 or config.D0.shape[0] != config.D0.shape[1]:
        raise ValueError(
            f"Example {config.example_id}: D0 must be a square matrix."
        )

    if config.D1.shape != config.D0.shape:
        raise ValueError(
            f"Example {config.example_id}: "
            "D1 must have the same shape as D0."
        )

    if not np.all(np.isfinite(config.D0)):
        raise ValueError(
            f"Example {config.example_id}: D0 contains non-finite values."
        )

    if not np.all(np.isfinite(config.D1)):
        raise ValueError(
            f"Example {config.example_id}: D1 contains non-finite values."
        )

    off_diagonal_d0 = config.D0.copy()
    np.fill_diagonal(off_diagonal_d0, 0.0)

    if np.any(off_diagonal_d0 < 0):
        raise ValueError(
            f"Example {config.example_id}: "
            "off-diagonal D0 entries must be non-negative."
        )

    if np.any(config.D1 < 0):
        raise ValueError(
            f"Example {config.example_id}: D1 entries must be non-negative."
        )

    if config.R_max < 1:
        raise ValueError(
            f"Example {config.example_id}: R_max must be at least 1."
        )

    if config.lam < 0:
        raise ValueError(
            f"Example {config.example_id}: lambda must be non-negative."
        )

    if config.decay_rate < 0:
        raise ValueError(
            f"Example {config.example_id}: decay_rate must be non-negative."
        )

    if config.t_end <= 0:
        raise ValueError(
            f"Example {config.example_id}: t_end must be positive."
        )

    if config.trajectories < 1:
        raise ValueError(
            f"Example {config.example_id}: trajectories must be at least 1."
        )


def build_burst_matrices(config: SSAExample) -> list[np.ndarray]:
    validate_config(config)

    return [
        (config.lam ** (r - 1)) * config.D1
        for r in range(1, config.R_max + 1)
    ]


for config in EXAMPLES.values():
    validate_config(config)

print("All nine example configurations passed validation.")

All nine example configurations passed validation.


In [6]:
def build_burst_model(
    config: SSAExample,
    initial_state: int = 0,
    initial_protein: int = 0,
) -> gillespy2.Model:
    validate_config(config)

    n_states = config.n_states

    if not 0 <= initial_state < n_states:
        raise ValueError(
            f"initial_state must be between 0 and {n_states - 1}; "
            f"received {initial_state}."
        )

    if initial_protein < 0:
        raise ValueError("initial_protein must be non-negative.")

    burst_matrices = build_burst_matrices(config)

    model = gillespy2.Model(
        name=f"Burst_Example_{config.example_id}"
    )

    switch_parameters: dict[
        tuple[int, int], gillespy2.Parameter
    ] = {}

    burst_parameters: dict[
        tuple[int, int, int], gillespy2.Parameter
    ] = {}

    all_parameters: list[gillespy2.Parameter] = []

    for i in range(n_states):
        for j in range(n_states):
            rate = float(config.D0[i, j])
            if i == j or rate == 0.0:
                continue

            parameter = gillespy2.Parameter(
                name=f"a{i + 1}_{j + 1}",
                expression=str(rate),
            )

            switch_parameters[(i, j)] = parameter
            all_parameters.append(parameter)

    for r, d_r in enumerate(burst_matrices, start=1):
        for i in range(n_states):
            for j in range(n_states):
                rate = float(d_r[i, j])

                if rate == 0.0:
                    continue

                parameter = gillespy2.Parameter(
                    name=f"b{i + 1}_{j + 1}_{r}",
                    expression=str(rate),
                )

                burst_parameters[(i, j, r)] = parameter
                all_parameters.append(parameter)

    decay_parameter = gillespy2.Parameter(
        name="delta",
        expression=str(float(config.decay_rate)),
    )
    all_parameters.append(decay_parameter)

    model.add_parameter(all_parameters)

    states = [
        gillespy2.Species(
            name=f"state{k + 1}",
            initial_value=1 if k == initial_state else 0,
        )
        for k in range(n_states)
    ]

    protein = gillespy2.Species(
        name="Protein",
        initial_value=int(initial_protein),
    )

    model.add_species([*states, protein])

    reactions: list[gillespy2.Reaction] = []

    for (i, j), parameter in switch_parameters.items():
        reactions.append(
            gillespy2.Reaction(
                name=f"A_{i + 1}_{j + 1}",
                reactants={states[i]: 1},
                products={states[j]: 1},
                rate=parameter.name,
            )
        )

    for (i, j, r), parameter in burst_parameters.items():
        reactions.append(
            gillespy2.Reaction(
                name=f"B_{i + 1}_{j + 1}_{r}",
                reactants={states[i]: 1},
                products={
                    states[j]: 1,
                    protein: r,
                },
                rate=parameter.name,
            )
        )
    reactions.append(
        gillespy2.Reaction(
            name="Protein_decay",
            reactants={protein: 1},
            products={},
            rate=decay_parameter.name,
        )
    )

    model.add_reaction(reactions)

    model.timespan(
        np.linspace(
            0.0,
            config.t_end,
            2,
        )
    )

    return model


In [7]:
@dataclass
class SimulationOutput:
    config: SSAExample
    protein_counts: np.ndarray
    elapsed_seconds: float
    output_path: Path
    model: gillespy2.Model
    raw_results: Any | None = None


def extract_final_protein_counts(results: Any) -> np.ndarray:
    counts = [
        int(trajectory["Protein"][-1])
        for trajectory in results
    ]

    return np.asarray(counts, dtype=np.int64)


def run_ssa(
    config: SSAExample,
    trajectory_override: int | None = None,
    output_dir: str | Path = ".",
    keep_raw_results: bool = False,
    **solver_run_kwargs: Any,
) -> SimulationOutput:
    validate_config(config)

    trajectories = (
        config.trajectories
        if trajectory_override is None
        else int(trajectory_override)
    )

    if trajectories < 1:
        raise ValueError("The trajectory count must be at least 1.")

    model = build_burst_model(config)
    solver = gillespy2.SSACSolver(model=model)

    start_time = time.perf_counter()

    results = solver.run(
        number_of_trajectories=trajectories,
        **solver_run_kwargs,
    )

    elapsed_seconds = time.perf_counter() - start_time
    protein_counts = extract_final_protein_counts(results)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / config.output_filename
    np.save(output_path, protein_counts)

    print(
        f"Example {config.example_id}: "
        f"{trajectories:,} trajectories completed in "
        f"{elapsed_seconds:.4f} seconds."
    )
    print(f"Saved final protein counts to: {output_path.resolve()}")

    return SimulationOutput(
        config=config,
        protein_counts=protein_counts,
        elapsed_seconds=elapsed_seconds,
        output_path=output_path,
        model=model,
        raw_results=results if keep_raw_results else None,
    )


def run_examples(
    example_ids: Iterable[int],
    trajectory_override: int | None = None,
    output_dir: str | Path = ".",
    keep_raw_results: bool = False,
    **solver_run_kwargs: Any,
) -> dict[int, SimulationOutput]:
    outputs: dict[int, SimulationOutput] = {}

    for example_id in example_ids:
        if example_id not in EXAMPLES:
            raise KeyError(
                f"Example {example_id} does not exist. "
                f"Valid identifiers are {sorted(EXAMPLES)}."
            )

        outputs[example_id] = run_ssa(
            config=EXAMPLES[example_id],
            trajectory_override=trajectory_override,
            output_dir=output_dir,
            keep_raw_results=keep_raw_results,
            **solver_run_kwargs,
        )

    return output


In [8]:
def plot_protein_distribution(
    protein_counts: np.ndarray,
    title: str = "Distribution of Protein Copy Number via SSA",
    save_path: str | Path | None = None,
    figure_size: tuple[float, float] = (10.0, 8.0),
    dpi: int = 150,
) -> tuple[plt.Figure, plt.Axes]:
    raw_counts = np.asarray(protein_counts)

    if raw_counts.ndim != 1:
        raise ValueError("protein_counts must be a one-dimensional array.")

    if raw_counts.size == 0:
        raise ValueError("protein_counts must not be empty.")

    if not np.issubdtype(raw_counts.dtype, np.number):
        raise TypeError("protein_counts must contain numeric values.")

    if not np.all(np.isfinite(raw_counts)):
        raise ValueError("protein_counts must contain only finite values.")

    if not np.all(raw_counts == np.floor(raw_counts)):
        raise ValueError("protein_counts must contain only integer values.")

    counts = raw_counts.astype(np.int64, copy=False)
    values, frequencies = np.unique(counts, return_counts=True)
    probabilities = frequencies / counts.size

    fig, ax = plt.subplots(
        figsize=figure_size,
        dpi=dpi,
        constrained_layout=True,
    )

    ax.bar(
        values,
        probabilities,
        width=0.8,
        label="Stochastic Simulation",
    )

    ax.set(
        title=title,
        xlabel="Number of Protein Molecules",
        ylabel="Probability",
        ylim=(0, None),
    )

    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.02)

    if save_path is not None:
        output_path = Path(save_path).expanduser()
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=dpi, bbox_inches="tight")

    return fig, ax

In [ ]:
# Keeping this False allows all notebook cells to execute
# without accidentally launching a large stochastic simulation.
RUN_SIMULATION = True

# Select one or more examples by identifier.
# SELECTED_EXAMPLE_IDS = [1]

# To select every example, use:
SELECTED_EXAMPLE_IDS = list(EXAMPLES)

# None means "use the original default for each example."
# A positive integer replaces the trajectory count for every selected example.
TRAJECTORY_OVERRIDE=None

# All .npy files produced by this notebook are placed in this directory.
OUTPUT_DIR = Path("ssa_outputs")

# This dictionary is populated only when RUN_SIMULATION is True.
simulation_outputs: dict[int, SimulationOutput] = {}

if RUN_SIMULATION:
    simulation_outputs = run_examples(
        example_ids=SELECTED_EXAMPLE_IDS,
        trajectory_override=TRAJECTORY_OVERRIDE,
        output_dir=OUTPUT_DIR,
    )
else:
    print(
        "Simulation is disabled. Set RUN_SIMULATION to True after reviewing "
        "SELECTED_EXAMPLE_IDS and TRAJECTORY_OVERRIDE."
    )
